# NL_cas_hl — candidate-action scoring under the corrected boundary

Seed 0, `bus14`, explicit-line graph, one-hop context **off**. Eight cells:
`mean`/`typed_mean` pooling x action features on/off (`f1`/`f0`) x dedicated
do-nothing head on/off (`a0h1`/`a0h0`).

Two sources: the full-test evaluation of each best checkpoint over the 201
held-out chronics, and the periodic 10-episode test curve from W&B.

**Read the marginals as descriptive only** — one seed per cell.

In [ ]:
import json, glob, os, re
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = "/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task"
FIG  = "/Users/corentinplumet/Documents/RL_Marl2grid/latex/figures"
CELL = re.compile(r"cas_hl_NL_(mean|tmean)_(f[01])_(a0h[01])_s0")

## Full-test evaluation of the best checkpoint

In [ ]:
# ---- full-test evaluation of the best checkpoint (201 held-out chronics) ----
rows = []
for f in sorted(glob.glob(f"{ROOT}/outputs/full_test_eval/NL_cas_hl/*.json")):
    d = json.load(open(f)); m = CELL.search(os.path.basename(f))
    rows.append(dict(pool=m.group(1), features=m.group(2), a0_head=m.group(3),
                     survival=d["survival_percent"], best_step=d["checkpoint_global_step"],
                     episodes=d["eval_episodes"]))
ft = pd.DataFrame(rows).sort_values("survival", ascending=False).reset_index(drop=True)
ft["cell"] = ft.pool + "/" + ft.features + "/" + ft.a0_head

## Training curves

In [ ]:
# ---- training curves ----
curves = {}
for d in sorted(glob.glob(f"{ROOT}/outputs/run_data/NL_cas_hl/runs/*")):
    m = CELL.search(os.path.basename(d))
    h = pd.read_parquet(os.path.join(d, "history.parquet"))
    s = h[["_step", "test/episodic_survival"]].dropna()
    curves[f"{m.group(1)}/{m.group(2)}/{m.group(3)}"] = s

## Figure 1 — test survival during training

In [ ]:
# ---- figure 1: training curves ----
fig, ax = plt.subplots(figsize=(9, 4.6))
cmap = plt.get_cmap("tab10")
for i, (name, s) in enumerate(sorted(curves.items())):
    w = s["test/episodic_survival"].rolling(9, min_periods=1, center=True).mean()
    ax.plot(s["_step"] / 1e6, 100 * w, color=cmap(i), lw=1.4, label=name)
ax.set_xlabel("environment steps (millions)")
ax.set_ylabel("test episodic survival (\\%)" if False else "test episodic survival (%)")
ax.set_ylim(0, 104); ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2, loc="lower right", framealpha=0.9)
ax.set_title("Candidate-action scoring under the corrected boundary, seed 0")
fig.tight_layout(); fig.savefig(f"{FIG}/nl_cas_hl_curves.png", dpi=200); plt.close(fig)

## Figure 2 — full-test survival by cell

In [ ]:
# ---- figure 2: full-test survival by cell ----
fig, ax = plt.subplots(figsize=(7.4, 4.0))
colors = ["#3b6ea5" if p == "mean" else "#b5651d" for p in ft.pool]
y = np.arange(len(ft))[::-1]
ax.barh(y, ft.survival, color=colors, height=0.62)
for yi, v in zip(y, ft.survival):
    ax.text(v + 0.8, yi, f"{v:.2f}", va="center", fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(ft.cell, fontsize=9)
ax.set_xlim(0, 108); ax.set_xlabel("full-test survival (%), 201 held-out chronics")
ax.axvline(98, color="grey", ls="--", lw=0.9)
ax.grid(axis="x", alpha=0.3)
handles = [plt.Rectangle((0,0),1,1,color="#3b6ea5"), plt.Rectangle((0,0),1,1,color="#b5651d")]
ax.legend(handles, ["mean pooling", "typed_mean pooling"], fontsize=9, loc="lower right")
ax.set_title("Best checkpoint, evaluated on the full test split")
fig.tight_layout(); fig.savefig(f"{FIG}/nl_cas_hl_fulltest.png", dpi=200); plt.close(fig)

print(ft[["cell", "survival", "best_step"]].to_string(index=False))
print("\nmarginals (single seed, so descriptive only):")
for col in ("pool", "features", "a0_head"):
    g = ft.groupby(col).survival.agg(["mean", "min", "max"]).round(2)
    print(g.to_string()); print()
print("cells at or above 98%:", int((ft.survival >= 98).sum()), "of", len(ft))